# Cloud Storage (GCS) Hands-On Lab — Google Cloud SDK
### GCP Training Program — Day 1, Module 1: Storage & Ingestion

**What this notebook covers:** every core Cloud Storage operation an instructor needs to demo live —
buckets, uploads/downloads, listing, copy/rename/delete, metadata, storage classes, lifecycle rules,
versioning, ACLs & IAM, signed URLs, CORS, and object composition — executed one cell at a time against
a real GCP project.

**Scope note:** this lab is scoped to **Cloud Storage** only. Spanner, Cloud SQL, Firestore, and
Bigtable are covered as separate hands-on labs elsewhere in Module 1.

**This notebook is fully self-contained.** Just run the cells top to bottom:
1. The **Authenticate** cell below will pop up a Google sign-in flow — sign in with the account
   that has access to your GCP project.
2. The **Configure** cell will *ask you* for your Project ID, region, and bucket name — nothing to
   hand-edit in code.
3. Everything after that runs against your own project automatically.

**Prerequisites (mention to the class before running):**
1. A GCP project with billing enabled.
2. IAM role on the project: `roles/storage.admin` (or equivalent) for the account you sign in with.
3. Python package `google-cloud-storage` (installed in a cell below).

**How to run in class:** execute cells top-to-bottom. Each section is self-contained but later
sections (lifecycle, versioning, ACLs) assume the bucket created in Section 4 still exists.


## 1. Authenticate This Session
Sign in with the Google account that has access to your GCP project. This works whether you're
running in **Google Colab** (pops up an interactive sign-in) or a local/Vertex Workbench kernel
(falls back to the Cloud SDK's Application Default Credentials).


In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab. This also configures the gcloud CLI for this session.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")
    print("If this is your first time, run in a terminal:")
    print("  gcloud auth login")
    print("  gcloud auth application-default login")

# Confirm which identity and project the Cloud SDK is using
!gcloud config list
!gcloud auth list

## 2. Install & Import the Client Library

In [ ]:
!pip install --quiet google-cloud-storage

In [ ]:
from google.cloud import storage
from datetime import timedelta
import time

print("google-cloud-storage imported successfully")

## 3. Configure Your Project, Region & Bucket
Enter your own values when prompted below — nothing to hand-edit in code. The bucket name gets a
timestamp suffix so it's globally unique (bucket names are global across all of GCS).

In [ ]:
import time

PROJECT_ID = input("Enter your GCP Project ID: ").strip()

REGION = input("Enter your region [default: asia-south1]: ").strip()
if not REGION:
    REGION = "asia-south1"

_default_bucket = f"gcs-training-demo-{int(time.time())}"
BUCKET_NAME = input(f"Enter a bucket name [default: {_default_bucket}]: ").strip()
if not BUCKET_NAME:
    BUCKET_NAME = _default_bucket

!gcloud config set project {PROJECT_ID}

client = storage.Client(project=PROJECT_ID)
print("Client initialized for project:", client.project)
print("Target bucket name:", BUCKET_NAME)

## 4. Bucket Operations
### 4.1 Create a Bucket

In [ ]:
bucket = client.bucket(BUCKET_NAME)
bucket.storage_class = "STANDARD"

new_bucket = client.create_bucket(bucket, location=REGION)
print(f"Created gs://{new_bucket.name} in {new_bucket.location} ({new_bucket.storage_class})")

# gsutil / Cloud SDK equivalent:
# gsutil mb -p PROJECT_ID -l asia-south1 -c STANDARD gs://BUCKET_NAME

### 4.2 List All Buckets in the Project

In [ ]:
print("Buckets in project", PROJECT_ID, ":")
for b in client.list_buckets():
    print(" -", b.name)

# gsutil equivalent: gsutil ls

### 4.3 Get Bucket Metadata

In [ ]:
bucket = client.get_bucket(BUCKET_NAME)

print("Name:              ", bucket.name)
print("Location:          ", bucket.location)
print("Storage class:     ", bucket.storage_class)
print("Created:           ", bucket.time_created)
print("Uniform bucket-level access:",
      bucket.iam_configuration.uniform_bucket_level_access_enabled)

# gsutil equivalent: gsutil ls -L -b gs://BUCKET_NAME

### 4.4 Update Bucket Configuration (labels + uniform access)

In [ ]:
bucket.labels = {"env": "training", "module": "gcs-lab"}
bucket.iam_configuration.uniform_bucket_level_access_enabled = True
bucket.patch()

print("Labels applied:", bucket.labels)
print("Uniform bucket-level access:", bucket.iam_configuration.uniform_bucket_level_access_enabled)

## 5. Object Upload / Download
### 5.1 Upload a Local File

In [ ]:
with open("sample.txt", "w") as f:
    f.write("Hello from the GCP Cloud Storage hands-on lab!\n")

blob = bucket.blob("uploads/sample.txt")
blob.upload_from_filename("sample.txt")
print(f"Uploaded to gs://{BUCKET_NAME}/{blob.name}")

# gsutil equivalent:
# gsutil cp sample.txt gs://BUCKET_NAME/uploads/sample.txt

### 5.2 Upload From In-Memory Data (no local file needed)

In [ ]:
blob2 = bucket.blob("uploads/inmemory.json")
blob2.upload_from_string('{"lab": "gcs", "day": 1}', content_type="application/json")
print("Uploaded in-memory object:", blob2.name)

### 5.3 Download an Object to a Local File

In [ ]:
blob = bucket.blob("uploads/sample.txt")
blob.download_to_filename("sample_downloaded.txt")

with open("sample_downloaded.txt") as f:
    print("Downloaded content ->", f.read())

# gsutil equivalent:
# gsutil cp gs://BUCKET_NAME/uploads/sample.txt sample_downloaded.txt

### 5.4 Download an Object Directly Into Memory

In [ ]:
content = bucket.blob("uploads/inmemory.json").download_as_text()
print("In-memory content:", content)

## 6. Listing & Organizing Objects
### 6.1 List All Objects in the Bucket

In [ ]:
for b in bucket.list_blobs():
    print(b.name, "-", b.size, "bytes")

### 6.2 List Objects Under a Prefix (simulated "folder")

In [ ]:
for b in bucket.list_blobs(prefix="uploads/"):
    print(b.name)

### 6.3 List With a Delimiter (one-level folder view)
`delimiter="/"` makes the API return top-level "folders" (prefixes) separately from top-level objects — this is exactly how the Cloud Console file browser simulates folders, since GCS has no real directories.

In [ ]:
iterator = bucket.list_blobs(prefix="", delimiter="/")
blobs = list(iterator)   # must consume the iterator before .prefixes is populated

print("Objects at root:   ", [b.name for b in blobs])
print("Prefixes (folders):", list(iterator.prefixes))

## 7. Copy, Rename, Move & Delete Objects
### 7.1 Copy an Object

In [ ]:
source_blob = bucket.blob("uploads/sample.txt")
copied_blob = bucket.copy_blob(source_blob, bucket, "uploads/sample_copy.txt")
print("Copied to:", copied_blob.name)

# gsutil equivalent:
# gsutil cp gs://BUCKET_NAME/uploads/sample.txt gs://BUCKET_NAME/uploads/sample_copy.txt

### 7.2 Rename / Move an Object
GCS has no native rename — `rename_blob` copies to the new name then deletes the original.

In [ ]:
renamed_blob = bucket.rename_blob(copied_blob, "archive/sample_renamed.txt")
print("Renamed/moved to:", renamed_blob.name)

### 7.3 Delete an Object

In [ ]:
bucket.blob("archive/sample_renamed.txt").delete()
print("Deleted archive/sample_renamed.txt")

# gsutil equivalent: gsutil rm gs://BUCKET_NAME/archive/sample_renamed.txt

## 8. Object Metadata
### 8.1 Read System Metadata

In [ ]:
blob = bucket.blob("uploads/sample.txt")
blob.reload()   # fetch latest metadata from GCS

print("Size:        ", blob.size)
print("Content-Type:", blob.content_type)
print("MD5 hash:    ", blob.md5_hash)
print("Generation:  ", blob.generation)
print("Updated:     ", blob.updated)

### 8.2 Set Custom Metadata & Content-Type

In [ ]:
blob.metadata = {"lab-owner": "instructor", "purpose": "gcs-demo"}
blob.content_type = "text/plain"
blob.patch()

print("Custom metadata now:", blob.metadata)

## 9. Storage Classes

| Class     | Min. storage duration | Typical access pattern      | Relative cost               |
|-----------|------------------------|------------------------------|------------------------------|
| STANDARD  | none                   | Frequently accessed / hot   | Highest storage, lowest retrieval |
| NEARLINE  | 30 days                | ~Monthly access              | Lower storage, small retrieval fee |
| COLDLINE  | 90 days                | ~Quarterly access            | Lower still, higher retrieval fee |
| ARCHIVE   | 365 days               | Rare / compliance archives   | Lowest storage, highest retrieval fee |

Good talking point for class: storage class is set **per object** or as a **bucket default** — changing
an object's class doesn't move it between locations, it's a metadata-level rewrite.


### 9.1 Change the Bucket's Default Storage Class

In [ ]:
bucket.storage_class = "NEARLINE"
bucket.patch()
print("Bucket default storage class is now:", bucket.storage_class)

### 9.2 Change the Storage Class of an Existing Object

In [ ]:
blob = bucket.blob("uploads/sample.txt")
blob.update_storage_class("COLDLINE")

refreshed = bucket.get_blob("uploads/sample.txt")
print("Object storage class is now:", refreshed.storage_class)

## 10. Object Lifecycle Management
### 10.1 Define & Apply Lifecycle Rules
Example policy: move objects to COLDLINE after 30 days, delete anything older than 365 days.

In [ ]:
rules = [
    {
        "action": {"type": "SetStorageClass", "storageClass": "COLDLINE"},
        "condition": {"age": 30},
    },
    {
        "action": {"type": "Delete"},
        "condition": {"age": 365, "isLive": True},
    },
]

bucket.lifecycle_rules = rules
bucket.patch()
print("Lifecycle rules applied.")

### 10.2 View the Current Lifecycle Configuration

In [ ]:
bucket.reload()
for rule in bucket.lifecycle_rules:
    print(rule)

# gsutil equivalent: gsutil lifecycle get gs://BUCKET_NAME

### 10.3 Remove All Lifecycle Rules

In [ ]:
bucket.clear_lifecycle_rules()
bucket.patch()
print("Lifecycle rules after removal:", bucket.lifecycle_rules)

## 11. Object Versioning
### 11.1 Enable Versioning

In [ ]:
bucket.versioning_enabled = True
bucket.patch()
print("Versioning enabled:", bucket.versioning_enabled)

### 11.2 Create Multiple Versions & List Generations
Each upload to the same object name creates a new **generation** instead of overwriting in place.

In [ ]:
blob = bucket.blob("uploads/versioned.txt")
blob.upload_from_string("version 1")
blob.upload_from_string("version 2")
blob.upload_from_string("version 3")

print("All generations of uploads/versioned.txt:")
for b in client.list_blobs(bucket, prefix="uploads/versioned.txt", versions=True):
    print(" generation:", b.generation, "| size:", b.size, "bytes")

### 11.3 Delete a Specific Version (generation)

In [ ]:
versions = list(client.list_blobs(bucket, prefix="uploads/versioned.txt", versions=True))
oldest = sorted(versions, key=lambda b: b.generation)[0]

bucket.blob(oldest.name, generation=oldest.generation).delete()
print("Deleted generation:", oldest.generation)

## 12. Access Control — ACLs & IAM
**Teaching point:** fine-grained ACLs and uniform bucket-level access (IAM-only) are mutually exclusive. We enabled uniform access in Section 3.4, so we disable it here to demo legacy ACLs, then show the modern IAM-based approach.

In [ ]:
bucket.iam_configuration.uniform_bucket_level_access_enabled = False
bucket.patch()
print("Uniform bucket-level access disabled - fine-grained ACLs are now usable.")

### 12.2 Grant Read Access to a Specific User via ACL

In [ ]:
blob = bucket.blob("uploads/sample.txt")
blob.acl.reload()
blob.acl.user("someone@example.com").grant_read()
blob.acl.save()
print("Granted READ on", blob.name, "to someone@example.com")

### 12.3 Make an Object Publicly Readable
**Use with caution in a live demo** — this exposes the object to the entire internet.

In [ ]:
blob.make_public()
print("Public URL:", blob.public_url)

### 12.4 IAM Policy at the Bucket Level (recommended over ACLs)

In [ ]:
policy = bucket.get_iam_policy(requested_policy_version=3)
policy.version = 3
policy.bindings.append({
    "role": "roles/storage.objectViewer",
    "members": {"group:data-readers@example.com"},
})
bucket.set_iam_policy(policy)
print("IAM policy updated on bucket:", bucket.name)

### 12.5 View the Current Bucket IAM Policy

In [ ]:
policy = bucket.get_iam_policy(requested_policy_version=3)
for binding in policy.bindings:
    print(binding["role"], "->", binding["members"])

# gsutil equivalent: gsutil iam get gs://BUCKET_NAME

## 13. Signed URLs — Temporary, Credential-less Access
**Note for class:** signing requires either a service-account key file or the `roles/iam.serviceAccountTokenCreator` role for impersonation. Plain end-user ADC credentials often can't sign directly — a common point of confusion worth calling out live.

In [ ]:
signed_get_url = blob.generate_signed_url(
    version="v4",
    expiration=timedelta(minutes=15),
    method="GET",
)
print("Signed GET URL (valid 15 min):\n", signed_get_url)

### 13.2 Signed URL for Uploading (PUT)

In [ ]:
upload_blob = bucket.blob("uploads/via_signed_url.txt")
signed_put_url = upload_blob.generate_signed_url(
    version="v4",
    expiration=timedelta(minutes=15),
    method="PUT",
    content_type="text/plain",
)
print("Signed PUT URL (valid 15 min):\n", signed_put_url)

## 14. CORS Configuration
Needed when a browser-based app on a different origin needs to call GCS directly (e.g. direct-to-bucket uploads).

In [ ]:
bucket.cors = [
    {
        "origin": ["https://example.com"],
        "method": ["GET", "HEAD"],
        "responseHeader": ["Content-Type"],
        "maxAgeSeconds": 3600,
    }
]
bucket.patch()
print("CORS configuration applied:", bucket.cors)

## 15. Compose Objects (server-side concatenation)
Combines multiple objects into one **without downloading and re-uploading** — useful after parallel chunked uploads.

In [ ]:
part1 = bucket.blob("compose/part1.txt")
part1.upload_from_string("Hello, ")

part2 = bucket.blob("compose/part2.txt")
part2.upload_from_string("World!")

composed = bucket.blob("compose/combined.txt")
composed.compose([part1, part2])

print("Composed object content:", composed.download_as_text())

## 16. Cleanup — Delete All Objects & the Bucket
Run this **last**, once every demo above has been shown, so the class sees teardown too.

In [ ]:
# delete every live object AND every archived version
for b in client.list_blobs(bucket, versions=True):
    bucket.blob(b.name, generation=b.generation).delete()

bucket.delete()
print(f"Bucket {BUCKET_NAME} and all contents deleted.")

# gsutil equivalent:
# gsutil rm -r gs://BUCKET_NAME